# 21 · Preparacion de datos — LABORATORY

**Orden (igual que prueba2, parando en encoding):**
```
Limpieza -> Imputacion (escalado + KNN) -> Encoding
```
Usa los mismos nodos Kedro (`src/nhanes/pipelines/...`) con el bloque `lab` de `parameters.yml`.

- **Input:** `data/02_intermediate/lab_intermediate.parquet` (SEQN = indice)
- **Target reservado:** 9 biomarcadores del PhenoAge (albumina, creatinina, glucosa, crp, linfocitos%, mcv, rdw, alp, wbc) -> edad biologica (Levine)
- **Nota:** `cotinina_cat` se elimina (redundante con `cotinina_log`). Sin categoricas -> sin One-Hot.
- **Salida:** `data/05_model_input/lab_encoded.parquet`

> El entrenamiento va despues.

In [1]:
import sys, yaml
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
from nhanes.pipelines.data_cleaning.nodes import clean_dataset
from nhanes.pipelines.data_imputation.nodes import impute_dataset
from nhanes.pipelines.data_encoding.nodes import encode_dataset

params = yaml.safe_load(open(ROOT / 'conf/base/parameters.yml', encoding='utf-8'))['lab']
print('Parametros lab:'); print(yaml.dump(params, allow_unicode=True, sort_keys=False))

df = pd.read_parquet(ROOT / 'data/02_intermediate/lab_intermediate.parquet')
print('Intermediate:', df.shape)
df.head()

Parametros lab:
cols_to_drop:
- cotinina_cat
target_cols:
- albumina
- creatinina
- glucosa_serica
- crp
- linfocitos_pct
- mcv
- rdw
- alp
- wbc
missing_threshold: 0.6
categorical_cols_impute: []
scaler_type: standard
knn_neighbors: 5
encoding:
  binary_cols: []
  ordinal_maps: {}
  nominal_cols: []

Intermediate: (8366, 38)


,alt,albumina,alp,bun,creatinina,globulina,glucosa_serica,hierro,ldh,bilirrubina_total,...,manganeso,ggt_log,acr_log,plomo_log,cadmio_log,cotinina_log,ast_alt_ratio,nlr,no_hdl,cotinina_cat
SEQN,,,,,,,,,,,,,,,,,,,,,
93703,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
93704,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,9.41,NaN,NaN,NaN,0.067659,NaN,NaN,0.891213,NaN,NaN
93705,16.0,4.4,74.0,11.0,0.92,2.9,85.0,94.0,174.0,0.6,...,8.57,3.091042,2.662355,1.381282,0.215111,0.027615,1.250000,1.220000,97.0,no_fumador
93706,10.0,4.4,79.0,12.0,0.81,2.7,94.0,163.0,142.0,0.7,...,14.07,2.833213,4.067145,0.553885,0.190620,0.129272,1.400000,2.495935,101.0,pasivo
93707,13.0,5.2,238.0,17.0,0.64,2.8,115.0,90.0,177.0,0.7,...,12.60,2.302585,3.039749,0.329304,0.131028,0.441476,1.846154,1.474394,121.0,pasivo


## 1. Limpieza (`data_cleaning`)

Elimina `cols_to_drop`, **reserva el target**, poda features con missingness > `missing_threshold`, y rellena categoricas faltantes con `'Unknown'`.

In [2]:
print('Nulos por columna ANTES:')
print(df.isnull().mean().round(3).sort_values(ascending=False).to_string())

lab_clean, lab_target_raw = clean_dataset(df, params)
print()
print('Features limpias:', lab_clean.shape)
print(list(lab_clean.columns))
print('Target reservado:', list(lab_target_raw.columns))
lab_target_raw.describe().round(2)

Nulos por columna ANTES:
ldh                  0.310
ast_alt_ratio        0.297
globulina            0.295
hierro               0.295
alt                  0.295
bun                  0.295
acido_urico          0.295
trigliceridos        0.295
ggt_log              0.295
glucosa_serica       0.295
calcio               0.295
creatinina           0.294
albumina             0.294
alp                  0.294
bilirrubina_total    0.294
hba1c                0.277
hdl                  0.195
no_hdl               0.195
plomo_log            0.177
cotinina_log         0.151
cotinina_cat         0.151
crp                  0.133
vitamina_d           0.114
manganeso            0.102
mercurio             0.102
selenio              0.102
cadmio_log           0.102
linfocitos_pct       0.101
monocitos_pct        0.101
neutrofilos_pct      0.101
nlr                  0.101
plaquetas            0.100
mpv                  0.100
mcv                  0.100
hemoglobina          0.100
wbc                  0.100
rdw

,albumina,creatinina,glucosa_serica,crp,linfocitos_pct,mcv,rdw,alp,wbc
count,5905.00,5903.00,5900.00,7250.00,7523.00,7528.00,7528.00,5903.00,7528.00
mean,4.08,0.88,100.79,3.44,34.11,86.56,13.78,90.62,7.38
std,0.35,0.45,34.16,7.41,10.66,6.59,1.27,52.39,5.11
min,2.10,0.25,47.00,0.11,4.40,35.40,11.30,16.00,1.90
25%,3.90,0.68,86.00,0.56,26.70,82.90,13.00,64.00,5.80
50%,4.10,0.82,92.00,1.36,33.10,87.10,13.50,79.00,7.00
75%,4.30,0.98,101.00,3.59,40.20,90.90,14.20,98.00,8.50
max,5.40,12.74,554.00,182.82,89.70,114.60,29.20,638.00,400.00


## 2. Imputacion (`data_imputation`)

`StandardScaler` y luego **KNN (k=5)** sobre los datos ya escalados (distancias comparables).

In [3]:
print('Nulos ANTES:', int(lab_clean.isnull().sum().sum()))
lab_imputed = impute_dataset(lab_clean, params)
print('Nulos DESPUES:', int(lab_imputed.isnull().sum().sum()))
lab_imputed.describe().round(3)

Nulos ANTES: 45727


Nulos DESPUES: 0


,alt,bun,globulina,hierro,ldh,bilirrubina_total,calcio,trigliceridos,acido_urico,monocitos_pct,...,selenio,manganeso,ggt_log,acr_log,plomo_log,cadmio_log,cotinina_log,ast_alt_ratio,nlr,no_hdl
count,8366.000,8366.000,8366.000,8366.000,8366.000,8366.000,8366.000,8366.000,8366.000,8366.000,...,8366.000,8366.000,8366.000,8366.000,8366.000,8366.000,8366.000,8366.000,8366.000,8366.000
mean,-0.048,-0.053,-0.005,-0.023,0.018,-0.043,0.023,-0.052,-0.072,-0.001,...,0.003,-0.000,-0.087,0.007,-0.038,0.004,-0.016,0.073,0.004,-0.046
std,0.866,0.873,0.876,0.875,0.872,0.871,0.872,0.867,0.884,0.956,...,0.956,0.955,0.886,0.959,0.928,0.956,0.935,0.897,0.957,0.927
min,-1.146,-2.108,-2.985,-2.110,-3.102,-1.286,-7.829,-1.030,-3.108,-3.359,...,-3.859,-2.325,-3.046,-2.168,-1.609,-0.837,-0.514,-1.995,-1.561,-2.576
25%,-0.497,-0.604,-0.527,-0.554,-0.493,-0.573,-0.482,-0.509,-0.677,-0.582,...,-0.586,-0.627,-0.676,-0.609,-0.686,-0.620,-0.514,-0.512,-0.569,-0.693
50%,-0.261,-0.170,-0.000,-0.090,-0.047,-0.216,-0.000,-0.242,-0.137,-0.089,...,-0.028,-0.111,-0.189,-0.192,-0.209,-0.251,-0.494,-0.000,-0.134,-0.141
75%,0.093,0.231,0.494,0.380,0.381,0.141,0.483,0.115,0.404,0.448,...,0.507,0.426,0.262,0.297,0.368,0.259,-0.000,0.530,0.338,0.431
max,23.518,10.760,6.754,10.615,17.681,11.557,6.382,25.527,6.549,21.947,...,10.151,11.081,5.967,6.585,8.406,9.389,3.276,10.907,16.583,6.939


## 3. Encoding (`data_encoding`)

Label / Ordinal / One-Hot segun el bloque `encoding`. Resultado totalmente numerico.

In [4]:
lab_encoded = encode_dataset(lab_imputed, params)
print('Shape final:', lab_encoded.shape)
print('dtypes:', set(lab_encoded.dtypes.astype(str)))
print('Nulos:', int(lab_encoded.isnull().sum().sum()), '| indice:', lab_encoded.index.name)
print('Columnas:', list(lab_encoded.columns))
lab_encoded.head()

Shape final: (8366, 28)
dtypes: {'float64'}
Nulos: 0 | indice: SEQN
Columnas: ['alt', 'bun', 'globulina', 'hierro', 'ldh', 'bilirrubina_total', 'calcio', 'trigliceridos', 'acido_urico', 'monocitos_pct', 'neutrofilos_pct', 'hemoglobina', 'plaquetas', 'mpv', 'hba1c', 'hdl', 'vitamina_d', 'mercurio', 'selenio', 'manganeso', 'ggt_log', 'acr_log', 'plomo_log', 'cadmio_log', 'cotinina_log', 'ast_alt_ratio', 'nlr', 'no_hdl']


,alt,bun,globulina,hierro,ldh,bilirrubina_total,calcio,trigliceridos,acido_urico,monocitos_pct,...,selenio,manganeso,ggt_log,acr_log,plomo_log,cadmio_log,cotinina_log,ast_alt_ratio,nlr,no_hdl
SEQN,,,,,,,,,,,,,,,,,,,,,
93703,-1.041375e-16,-1.240229e-16,-1.042153e-15,-8.969067e-17,-1.673892e-16,-1.901842e-16,-1.552092e-15,-6.080733e-17,-9.632845e-17,-2.054274e-16,...,-1.834757e-16,1.867858e-16,-6.320484e-16,7.541138e-17,-1.238599e-17,1.144359e-16,-1.450904e-17,-3.225347e-16,-9.350489e-17,-7.592621e-17
93704,-3.906181e-01,-3.601221e-02,-2.021362e-01,-4.719429e-01,-4.186009e-01,-1.143325e+00,-3.212069e-01,1.682755e-01,-1.163159e+00,-8.936466e-02,...,-1.671590e+00,-2.408757e-01,-7.838898e-01,-1.623173e-01,-7.369984e-01,-8.369812e-01,-3.154020e-01,6.417711e-01,-8.654034e-01,-5.941006e-01
93705,-3.198133e-01,-6.042155e-01,-4.340190e-01,1.834470e-01,4.093186e-01,4.976679e-01,-3.212069e-01,-3.889025e-01,2.684990e-01,-3.581017e-01,...,-2.236671e-02,-4.641667e-01,-4.594196e-02,1.566242e-01,1.974893e+00,-2.511003e-01,-5.054806e-01,1.523058e-01,-5.785014e-01,-7.272071e-01
93706,-6.738373e-01,-4.370969e-01,-8.977848e-01,2.067693e+00,-5.042477e-01,8.544055e-01,7.513307e-01,-4.163948e-01,1.754181e+00,4.033198e-01,...,4.990202e-01,9.978577e-01,-4.341920e-01,1.503927e+00,-2.504195e-01,-3.484117e-01,-4.532733e-01,4.919319e-01,5.348906e-01,-6.286097e-01
93707,-4.968253e-01,3.984962e-01,-6.659019e-01,7.421533e-02,4.949655e-01,8.544055e-01,2.092003e+00,-2.514408e-01,6.590597e-02,-8.955756e-01,...,-2.642629e-01,6.070984e-01,-1.233235e+00,5.185745e-01,-8.544388e-01,-5.851917e-01,-2.929377e-01,1.502102e+00,-3.565154e-01,-1.356227e-01


## 4. Guardado

In [5]:
for d in ['03_primary','04_feature','05_model_input']:
    (ROOT / 'data' / d).mkdir(parents=True, exist_ok=True)
lab_clean.to_parquet(ROOT / 'data/03_primary/lab_clean.parquet')
lab_target_raw.to_parquet(ROOT / 'data/03_primary/lab_target_raw.parquet')
lab_imputed.to_parquet(ROOT / 'data/04_feature/lab_imputed.parquet')
lab_encoded.to_parquet(ROOT / 'data/05_model_input/lab_encoded.parquet')
print('Guardado OK')

Guardado OK


## Conclusiones — LABORATORY

- Los 9 biomarcadores quedan reservados como target (la edad biologica PhenoAge se calcula en modelado con la edad de DEMO). El resto (metales, lipidos, ratios, etc.) son features.
- Salida `lab_encoded`: escalado, sin nulos, `SEQN` como indice para unir con DEMO al final.
- **Pendiente (modelado):** construir target de clasificacion, split y entrenar (reg / clf / no supervisado).